In [1]:
!pip install rank_bm25 sentence-transformers faster-whisper google-search-results ipywidgets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 133.2 MB/s eta 0:00:00
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=07dfee057dc01febff7caadbca753ef2064a80f99a67d76780e882211ec2dede
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [2]:
import pandas as pd
import numpy as np
import re
import time
from collections import Counter, defaultdict

import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, HTML, Audio, Javascript, clear_output
import ipywidgets as widgets

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
drake = pd.read_csv("drake_data.csv")
billie = pd.read_csv("BillieEilish.csv")
maroon = pd.read_csv("Maroon5.csv")

In [4]:
# Drake
drake["artist"] = "Drake"
drake = drake.rename(columns={
    "lyrics_title": "song_name",
    "lyrics": "lyrics",
    "album": "album",
    "track_views": "track_views",
    "lyrics_url": "lyrics_url"
})
drake = drake[["artist", "song_name", "album", "lyrics", "track_views", "lyrics_url"]]

# Billie
billie = billie.rename(columns={
    "Artist": "artist",
    "Title": "song_name",
    "Album": "album",
    "Lyric": "lyrics"
})
billie["track_views"] = "Not available"
billie["lyrics_url"] = "Not available"
billie = billie[["artist", "song_name", "album", "lyrics", "track_views", "lyrics_url"]]

# Maroon 5
maroon = maroon.rename(columns={
    "Artist": "artist",
    "Title": "song_name",
    "Album": "album",
    "Lyric": "lyrics"
})
maroon["track_views"] = "Not available"
maroon["lyrics_url"] = "Not available"
maroon = maroon[["artist", "song_name", "album", "lyrics", "track_views", "lyrics_url"]]

# Merge
df = pd.concat([drake, billie, maroon], ignore_index=True)

df = df.dropna(subset=["lyrics"])
df = df.drop_duplicates(subset=["artist", "song_name"])
df = df.reset_index(drop=True)

df["doc_id"] = df.index

print("Total songs:", len(df))
df.head()

Total songs: 618


,artist,song_name,album,lyrics,track_views,lyrics_url,doc_id
0,Drake,Certified Lover Boy* Lyrics,Certified Lover Boy,[Verse]\nPut my feelings on ice\nAlways been a...,8.7K,https://genius.com/Drake-certified-lover-boy-l...,0
1,Drake,Like I’m Supposed To/Do Things Lyrics,Certified Lover Boy,[Verse]\nHands are tied\nSomeone's in my ear f...,38.8K,https://genius.com/Drake-like-im-supposed-to-d...,1
2,Drake,Not Around Lyrics,Certified Lover Boy,"[Intro]\nYeah, we back\nWassup ladies?\nSwisha...",129.8K,https://genius.com/Drake-not-around-lyrics,2
3,Drake,In the Cut (Ft. Roddy Ricch) Lyrics,Certified Lover Boy,"[Intro: Drake]\nAyy, yeah\nPipe this shit up a...",72.1K,https://genius.com/Drake-in-the-cut-lyrics,3
4,Drake,Zodiac Sign (Ft. Jessie Reyez) Lyrics,Certified Lover Boy,[Verse 1: Drake]\nYou ask how many girls I bee...,54.8K,https://genius.com/Drake-zodiac-sign-lyrics,4


Text cleaning


In [5]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text)
    text = re.sub(r"\[.*?\]", " ", text)      # remove [Verse], [Chorus]
    text = text.replace("Embed", " ")
    text = text.replace("You might also like", " ")
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_for_bm25(text):
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

df["clean_lyrics"] = df["lyrics"].apply(clean_text)
df["bm25_tokens"] = df["lyrics"].apply(preprocess_for_bm25)

df[["artist", "song_name", "clean_lyrics"]].head()

,artist,song_name,clean_lyrics
0,Drake,Certified Lover Boy* Lyrics,put my feelings on ice always been a gem certi...
1,Drake,Like I’m Supposed To/Do Things Lyrics,hands are tied someone s in my ear from the ot...
2,Drake,Not Around Lyrics,yeah we back wassup ladies swishahouse baby wa...
3,Drake,In the Cut (Ft. Roddy Ricch) Lyrics,ayy yeah pipe this shit up and i turn this shi...
4,Drake,Zodiac Sign (Ft. Jessie Reyez) Lyrics,you ask how many girls i been with in my life ...


BM25

In [6]:
bm25_model = BM25Okapi(df["bm25_tokens"].tolist())

print("BM25 model is ready")

BM25 model is ready


Load Sentence-BERT

In [7]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence-BERT model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT model loaded


lyrics embeddings

In [8]:
lyrics_embeddings = sbert_model.encode(
    df["clean_lyrics"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Lyrics embeddings shape:", lyrics_embeddings.shape)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Lyrics embeddings shape: (618, 384)


BM25 candidate retrieval

In [9]:
def bm25_retrieve_candidates(query, top_n=20):
    query_tokens = preprocess_for_bm25(query)
    scores = bm25_model.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_n]

    candidates = df.iloc[top_indices].copy()
    candidates["bm25_score"] = scores[top_indices]

    return candidates

Sentence-BERT re-ranking

In [10]:
def sbert_rerank(query, candidates, top_k=5):
    query_clean = clean_text(query)

    query_embedding = sbert_model.encode(
        [query_clean],
        convert_to_numpy=True
    )

    candidate_indices = candidates.index.tolist()
    candidate_embeddings = lyrics_embeddings[candidate_indices]

    sbert_scores = cosine_similarity(
        query_embedding,
        candidate_embeddings
    ).flatten()

    results = candidates.copy()
    results["sbert_score"] = sbert_scores

    results = results.sort_values(
        by="sbert_score",
        ascending=False
    ).head(top_k)

    return results

Full hybrid search function

In [11]:
def hybrid_search(query, bm25_top_n=20, final_top_k=5):
    start_time = time.time()

    candidates = bm25_retrieve_candidates(query, top_n=bm25_top_n)
    results = sbert_rerank(query, candidates, top_k=final_top_k)

    end_time = time.time()
    search_time = round(end_time - start_time, 4)

    return results, search_time

SerpAPI setup

In [12]:
from serpapi import GoogleSearch

SERPAPI_KEY = "e956a3dbccd7d4f90be734f8d55b1d07"


Search song links using SerpAPI

In [13]:
def search_song_links_serpapi(song_name, artist):
    params = {
        "engine": "google",
        "q": f"{song_name} {artist} Spotify Genius YouTube",
        "api_key": SERPAPI_KEY
    }

    search = GoogleSearch(params)
    data = search.get_dict()

    links = []

    for item in data.get("organic_results", [])[:5]:
        links.append({
            "title": item.get("title"),
            "link": item.get("link"),
            "snippet": item.get("snippet")
        })

    return pd.DataFrame(links)

Audio recording function

In [15]:
from google.colab import output
from base64 import b64decode
from faster_whisper import WhisperModel

def get_audio_query(record_seconds=8):
    filename = "recorded.wav"

    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {record_seconds * 1000}));

        recorder.stop();

        await new Promise(resolve => recorder.onstop = resolve);

        const blob = new Blob(chunks);
        const arrayBuffer = await blob.arrayBuffer();
        const bytes = new Uint8Array(arrayBuffer);

        let binary = '';
        const chunkSize = 8192;

        for (let i = 0; i < bytes.length; i += chunkSize) {{
            let subarray = bytes.subarray(i, i + chunkSize);
            binary += String.fromCharCode.apply(null, subarray);
        }}

        return btoa(binary);
    }}

    recordAudio();
    """

    audio_base64 = output.eval_js(js_code)
    audio_bytes = b64decode(audio_base64)

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    print("Recording saved.")

    whisper_model = WhisperModel("medium", compute_type="int8")

    segments, info = whisper_model.transcribe(
        filename,
        language="en",
        beam_size=5
    )

    raw_text = " ".join([seg.text for seg in segments])
    query = clean_text(raw_text)

    print("Raw text:", raw_text)
    print("Clean query:", query)

    return query

UI

In [28]:
selected_input = None
user_query = ""

title = widgets.HTML("<h2>🎵 Hybrid Lyrics Search Engine: BM25 + Sentence-BERT</h2>")

input_label = widgets.HTML("<b>Choose input type:</b>")

text_button = widgets.Button(description="Text Input", button_style="info")
audio_button = widgets.Button(description="Audio Input", button_style="warning")

query_box = widgets.Text(
    placeholder="Enter part of lyrics here...",
    description="Lyrics:",
    layout=widgets.Layout(width="90%")
)

search_button = widgets.Button(description="Search", button_style="danger")

output_box = widgets.Output()

def choose_text(b):
    global selected_input
    selected_input = "text"

    with output_box:
        clear_output()
        print("Text input selected. Write lyrics in the box, then click Search.")

def choose_audio(b):
    global selected_input
    selected_input = "audio"

    with output_box:
        clear_output()
        print("Audio input selected. Click Search, then allow microphone permission.")

def run_final_search(b):
    global user_query

    with output_box:
        clear_output()

        if selected_input is None:
            print("Please choose Text Input or Audio Input first.")
            return

        if selected_input == "text":
            user_query = query_box.value

            if user_query.strip() == "":
                print("Please enter lyrics first.")
                return

        elif selected_input == "audio":
            print("Recording will start now...")
            user_query = get_audio_query(record_seconds=8)

            if user_query.strip() == "":
                print("No audio text detected. Please try again.")
                return

        print("Query:", user_query)
        print("\nSearching using BM25 + Sentence-BERT...")

        results, search_time = hybrid_search(user_query)

        if len(results) == 0:
            print("No result found.")
            return

        best = results.iloc[0]

        print("\nBest Matching Song")
        print("-------------------")
        print("Artist:", best["artist"])
        print("Song Name:", best["song_name"])
        print("Album:", best["album"])
        print("BM25 Score:", round(best["bm25_score"], 4))
        print("Sentence-BERT Score:", round(best["sbert_score"], 4))
        print("Search Time:", search_time, "seconds")
        print("Lyrics URL:", best["lyrics_url"])

        print("\nTop 5 Results:")
        display(results[[
            "artist",
            "song_name",
            "album",
            "bm25_score",
            "sbert_score",
            "lyrics_url"
        ]])

        print("\nExternal Links from SerpAPI:")
        links = search_song_links_serpapi(
            best["song_name"],
            best["artist"]
        )
        display(links)

text_button.on_click(choose_text)
audio_button.on_click(choose_audio)
search_button.on_click(run_final_search)

display(title)
display(input_label)
display(widgets.HBox([text_button, audio_button]))
display(query_box)
display(search_button)
display(output_box)

HTML(value='<h2>🎵 Hybrid Lyrics Search Engine: BM25 + Sentence-BERT</h2>')

HTML(value='<b>Choose input type:</b>')

Text(value='', description='Lyrics:', layout=Layout(width='90%'), placeholder='Enter part of lyrics here...')

Button(button_style='danger', description='Search', style=ButtonStyle())

Output()